In [86]:

# External imports
import jax
import jax.numpy as jnp
from jax import config

config.update("jax_enable_x64", True)

import matplotlib.pyplot as plt
# Internal imports
from mGST.low_level_jit import cost_function_jax_mps
from mGST.trust_region import riemannian_gradient_fn, riemannian_hessian_vector_fn

from mGST.utility_functions_comparisons import get_full_mgst_parameters_from_configuration, GSTConfiguration, get_isometry_dimensions_from_tensor, tensor_to_isometry, isometry_to_tensor, get_compressed_rep_from_mgst_output
from mGST.riemannian import is_isometry, is_in_tangent_space, riemannian_metric, is_in_normal_space, random_normal_vector, random_tangent_vector, project_onto_normal_space, project_onto_tangent_space
from mGST.linear_algebra import random_anti_hermitian_matrix

backend = "iqmfakeapollo"

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Goals:

**Final goal:** Find out how (if at all) are the Hessian rank and the Gauge directions related.

We are currently looking at the action of the Gauge along the Kraus (rank) leg. For this purpose we assume the group acting as a Gauge on this leg is only unitary $U(r)$. From this, we can study this action in terms of a curve on the manifold, and then, associate the tangent space of the curve with the Horizontal space (subspace of the full tangent space).

Concretely, the gauge curve at $X \in \mathrm{St}(n,p)$ is:

\begin{equation}
c(t) = \left(\mathbb{I} \otimes \exp(A * t)\right) X, \quad  A^\dagger = -A \in \mathbb{C}^{r \times r} (\text{Anti-Hermitian})
\end{equation}

Therefore, the derivative of the curve is

\begin{equation}
c'(t) = \left(\mathbb{I} \otimes A\exp(A * t)\right) X
\end{equation}

and its velocity ($t = 0$) 

\begin{equation}
v := c'(0) = \left(\mathbb{I} \otimes A\right) X
\end{equation}

Similarly, its acceleration at t = 0 is then:

\begin{equation}
a := c''(0) = \left(\mathbb{I} \otimes A^2\right) X
\end{equation}

**Reminder**: $(A^2)^\dagger = A^2$ (Hermitian)

All together, can be used to write the second order taylor expansion of $g:= f \circ c : \R \to \R$ as:

\begin{equation}
f(c(t))= f(X) + t \langle\mathrm{grad}f(X), v \rangle_X + \frac{t^2}{2} \left( \langle \mathrm{Hess}f(X)[v], v \rangle_X  + \langle \mathrm{grad}f(X), c''(0)  \rangle_X\right)
\end{equation}

Check `numerically`:

1. Is $\langle\mathrm{grad}f(X), v \rangle_X = 0$? If so, this means the gradient is fully in the horizontal space (since it's orthogonal to a vector that is by definition in the vertical space).

2. Similarly, is $\langle \mathrm{grad}f(X), c''(0)  \rangle_X = 0$? Together with (1), this would mean the acceleration is on the vertical space (is this always true?).

3. Lastly, is $\langle \mathrm{Hess}f(X)[v], v \rangle_X = 0$?

> *NOTE*: recall that in our tensor ordering for Kraus, we have `(Kraus_rank, dim_out, dim_in)`, which means that we have to switch the order of the Identity and A in the expressions above!! (IMPORTANT)

Notes:
1. See Lemma 9.41 for a possibly important discussion relating Hessian spectrum and vertical space. https://www.nicolasboumal.net/book/IntroOptimManifolds_Boumal_2023.pdf



# Tensor Generation

In [3]:
dim = 2**2
num_circuits = 800
shots = 1000
kraus_rank = dim**2

Q2_GST = GSTConfiguration(
    qubit_layouts=[[0, 1]],
    gate_set="2QXYICZ",
    num_circuits=num_circuits,
    shots=shots,
    rank=kraus_rank,
)

kraus_tensor_mgst_2q, kraus_mgst_2q, povm_mgst_2q, state_mgst_2q, prob_matrix_2q, indices_list_2q = get_full_mgst_parameters_from_configuration(
    Q2_GST, backend, only_jax_variables=True
)

kraus_tensor_2q_target, povm_psd_2q_target, state_psd_2q_target = get_compressed_rep_from_mgst_output(kraus_mgst_2q, povm_mgst_2q, state_mgst_2q, kraus_rank=kraus_rank, state_rank=dim, povm_rank=dim)

2026-02-10 17:07:43,173 - iqm.benchmarks.logging_config - INFO - Generating 800 random GST circuits
2026-02-10 17:07:43,592 - iqm.benchmarks.logging_config - INFO - Will transpile all 800 circuits according to fixed physical layout
2026-02-10 17:07:43,909 - iqm.benchmarks.logging_config - INFO - Transpiling for backend IQMFakeApolloBackend with optimization level 0, sabre routing method all circuits
2026-02-10 17:07:46,147 - iqm.benchmarks.logging_config - INFO - Submitting batch with 800 circuits corresponding to qubits [0, 1]
2026-02-10 17:07:46,161 - iqm.benchmarks.logging_config - INFO - Stored jobs for 1 layouts in dataset
2026-02-10 17:07:46,223 - iqm.benchmarks.logging_config - INFO - Now executing the corresponding circuit batch
2026-02-10 17:07:46,248 - iqm.benchmarks.logging_config - INFO - Retrieving all counts
2026-02-10 17:07:47,402 - iqm.benchmarks.logging_config - INFO - Adding counts to dataset
2026-02-10 17:07:48,988 - iqm.benchmarks.logging_config - INFO - Run complet

# Compute $X$ and $\mathrm{rGrad}f(x)$ as $\mathbb{C}^{n \times p}$ operators

In [4]:
# Define a cost function only for Kraus
cost_fn_kraus = lambda x: cost_function_jax_mps(
    kraus_tensor=x,
    povm_psd=povm_psd_2q_target,
    state_psd=state_psd_2q_target,
    prob_matrix=prob_matrix_2q,
    indices_list=indices_list_2q,
    jit=True
)

In [5]:
cost_x = cost_fn_kraus(kraus_tensor_2q_target)
print(cost_x)

at iteration 0 the new count changed to: 1
at iteration 1 the new count changed to: 2
at iteration 7 the new count changed to: 3
at iteration 43 the new count changed to: 4
at iteration 102 the new count changed to: 5
at iteration 161 the new count changed to: 6
at iteration 220 the new count changed to: 7
at iteration 280 the new count changed to: 8
at iteration 340 the new count changed to: 9
at iteration 400 the new count changed to: 10
at iteration 466 the new count changed to: 11
at iteration 532 the new count changed to: 12
at iteration 599 the new count changed to: 13
at iteration 666 the new count changed to: 14
at iteration 733 the new count changed to: 15
0.0015532254504062397


In [94]:
# Obtain Riemannian gradient
metric = "euclidean"
grad_fx = riemannian_gradient_fn(x=kraus_tensor_2q_target, cost_fn=cost_fn_kraus, operator_type="kraus", metric=metric)

In [95]:
kraus_tensor_2q_target.shape, jnp.linalg.norm(grad_fx) # non-zero gradient. That's good.

((6, 16, 4, 4), Array(0.01091343, dtype=float64))

In [96]:
# obtain the (n,p) operators from the kraus tensor and it's riemannian gradient 
n, p = get_isometry_dimensions_from_tensor(kraus_tensor_2q_target, operator_type="kraus")
single_kraus = kraus_tensor_2q_target[0, :, :, :]
x = tensor_to_isometry(single_kraus, n, p)

single_grad_fx = grad_fx[0, :, :, :]
rgrad_x = tensor_to_isometry(single_grad_fx, n, p)

In [97]:
# Check X is an isometry
print(is_isometry(x), x.shape)

# Check the gradient is in the tangent space
print(is_in_tangent_space(x=x, z=rgrad_x))

True (64, 4)
True


# Compute the velocity and acceleration of curve

In [98]:
# Fist, let's validate that following this curve for any t we obtain the same cost value.
def curve(t, A: jnp.ndarray, x:jnp.ndarray):
    return jnp.kron(jax.scipy.linalg.expm(t * A), jnp.eye(dim)) @ x

def single_isometry_back_to_kraus_tensor(isometry:jnp.ndarray):
    x_kraus = isometry_to_tensor(isometry=isometry, tensor_shape=single_kraus.shape) # (kraus_rank, dim, dim)
    # make a copy of original full kraus tensor and replace the first element with the one obtained from the isometry
    kraus_tensor = jnp.copy(kraus_tensor_2q_target)
    return kraus_tensor.at[0, :, :, :].set(x_kraus)
    
# random anti-hermitian matrix
I_dim = jnp.eye(dim) # Identity accompaying the remaining of the n dimension (dim)
A = random_anti_hermitian_matrix(n=kraus_rank, seed=42)
jnp.allclose(-A, A.conj().T)

Array(True, dtype=bool)

In [99]:
# Check if the curve produces valid isometries
t_values = jnp.linspace(0, 10, 21)
isometries_along_curve = [curve(t, A, x) for t in t_values]
all([is_isometry(isometry) for isometry in isometries_along_curve]) # all True, as expected

True

In [100]:
t_values

Array([ 0. ,  0.5,  1. ,  1.5,  2. ,  2.5,  3. ,  3.5,  4. ,  4.5,  5. ,
        5.5,  6. ,  6.5,  7. ,  7.5,  8. ,  8.5,  9. ,  9.5, 10. ],      dtype=float64)

In [101]:
# Are they the same to each other?
jnp.allclose(isometries_along_curve[0], isometries_along_curve[1]) # False, as expected

Array(False, dtype=bool)

In [102]:
# verify the cost function along all of them is the same
kraus_tensor_along_curve = [single_isometry_back_to_kraus_tensor(isometry) for isometry in isometries_along_curve]
costs_along_curve = [cost_fn_kraus(kraus_tensor) for kraus_tensor in kraus_tensor_along_curve]
print("Are all costs along the curve the same: ", all(jnp.isclose(cost_curve, cost_x) for cost_curve in costs_along_curve)) # all the same, as expected, since they are all gauge transformations of the same point in the manifold

Are all costs along the curve the same:  True


In [103]:
# velocity
velocity = jnp.kron(A, I_dim) @ x
is_in_tangent_space(x=x, z=velocity), is_in_normal_space(x=x, z=velocity) 

(Array(True, dtype=bool), Array(False, dtype=bool))

In [104]:
# acceleration
acceleration = jnp.kron(A @ A, I_dim) @ x
is_in_tangent_space(x=x, z=acceleration), is_in_normal_space(x=x, z=acceleration) # True, since A^2 is hermitian, the normal space condition is satisfied

(Array(False, dtype=bool), Array(True, dtype=bool))

In [105]:
acceleration_projected_normal = project_onto_normal_space(x=x, z=acceleration)
jnp.allclose(acceleration, acceleration_projected_normal) # they are NOT the same

Array(False, dtype=bool)

In [106]:
acceleration_projected_tangent = project_onto_tangent_space(x=x, z=acceleration)
jnp.allclose(acceleration, acceleration_projected_tangent) # they are NOT the same

Array(False, dtype=bool)

In [107]:
jnp.linalg.norm(acceleration_projected_normal - acceleration)

Array(26.29282975, dtype=float64)

In [49]:
# Compute the Riemannian Hessian in the direction of the velocity

# first, let's create a tangent vector tensor with the same dimensions as the kraus tensor: (num_gates, kraus_rank, dim, dim)
velocity_tensor = isometry_to_tensor(isometry=velocity, tensor_shape=single_kraus.shape) # (kraus_rank, dim, dim)
velocity_full_tensor = jnp.broadcast_to(velocity_tensor, kraus_tensor_2q_target.shape) # (num_gates, kraus_rank, dim, dim)
jnp.allclose(velocity_full_tensor[2], velocity_tensor)

Array(True, dtype=bool)

In [111]:
# Now let's compute the rhessianf(x)[velocity]
rhessian_x_to_v = riemannian_hessian_vector_fn(x=kraus_tensor_2q_target, tangent_vector=velocity_full_tensor, cost_fn=cost_fn_kraus, operator_type="kraus", metric="euclidean")

In [112]:
rhessian_x_to_v_single = rhessian_x_to_v[0, :, :, :]
rHv_isometry = tensor_to_isometry(rhessian_x_to_v_single, n, p)
is_in_tangent_space(x=x, z=rHv_isometry), is_in_normal_space(x=x, z=rHv_isometry)

(Array(True, dtype=bool), Array(False, dtype=bool))

In [113]:
jnp.linalg.norm(rHv_isometry) # not zero, good!

Array(0.04029028, dtype=float64)

# Compute Riemannian inner product

In [61]:
# First for gradient and velocity
rgrad_velocity_inner_product = riemannian_metric(z1 = rgrad_x, z2=velocity, x=x, metric=metric)
print("<gradf(x), v>_x: ", rgrad_velocity_inner_product)

<gradf(x), v>_x:  6.623797647528629e-19


In [114]:
# Now for the acceleration
rgrad_acceleration_inner_product = riemannian_metric(z1 = rgrad_x, z2=acceleration, x=x, metric=metric) 
print("<gradf(x), a>_x: ", rgrad_acceleration_inner_product)
# Why is this not more zero?

<gradf(x), a>_x:  0.0001310243887273475


In [115]:
# Now for the acceleration
rgrad_acceleration_inner_product_tx = riemannian_metric(z1 = rgrad_x, z2=acceleration_projected_tangent, x=x, metric=metric) 
print("<gradf(x), a>_x: ", rgrad_acceleration_inner_product_tx)
# Why is this not more zero?

<gradf(x), a>_x:  0.00013102438872647623


In [ ]:
# Now for Hessian and velocity
rhessian_velocity_inner_product = riemannian_metric(z1 = rHv_isometry, z2=velocity, x=x, metric="euclidean")
print("<Hessf(x)[v], v>_x: ", rhessian_velocity_inner_product) # this is using the euclidean metric

<Hessf(x)[v], v>_x:  -0.00013102438872748985


In [72]:
# Now for Hessian and velocity
rhessian_velocity_inner_product = riemannian_metric(z1 = rHv_isometry, z2=velocity, x=x, metric=metric)
print("<Hessf(x)[v], v>_x: ", rhessian_velocity_inner_product)

<Hessf(x)[v], v>_x:  0.00015474457770418143


# Checking: inner product between random tangent and normal vectors

In [84]:
z_tangent = random_tangent_vector(x=x, seed=40)
z_normal = random_normal_vector(x=x, seed=41)
print(is_in_tangent_space(x=x, z=z_tangent), is_in_normal_space(x=x, z=z_tangent))
print(is_in_tangent_space(x=x, z=z_normal), is_in_normal_space(x=x, z=z_normal))

True False
False True


In [85]:
riemannian_metric(z1 = z_tangent, z2=z_normal, x=x, metric="canonical") # should be close to zero

Array(-7.77156117e-16, dtype=float64)